<div style="display:flex; align-items:center; gap:18px; text-align:left">

<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Semana 12: Reproducibilidad y entrega técnica</p>
</div>
</div>

# 12 · Reproducibilidad y entrega técnica

> **La pregunta de hoy:** ¿cómo se pasa de varios modelos entrenados a una recomendación que se pueda
> defender con cifras?

Este cuaderno es la **plantilla del taller**, resuelta con el caso de cobranza: **armar el pipeline**,
**la tabla de comparación** y **guardar el elegido con joblib**. La elección con su motivo, la prueba y
la recomendación se trabajan en las slides. No se envía: sirve para obtener las cifras que van a la
presentación del proyecto.

## Preparación del entorno

En Colab estas librerías ya vienen instaladas; la celda queda para correr fuera de Colab. La siguiente
define `num`, que escribe las cifras con punto de miles y coma decimal.

In [1]:
%%capture
!pip install -q pandas numpy scikit-learn joblib

In [2]:
from IPython.display import Markdown, display  # muestra frases con las cifras ya calculadas

def num(x, dec=0):
    # miles con punto, decimales con coma y el menos tipográfico
    texto = f'{x:,.{dec}f}'.replace(',', '@').replace('.', ',').replace('@', '.')
    return texto.replace('-', '−')

## Cargar datos

La cartera de cobranza de **tarjetas de crédito**, `riesgo_impago.csv`, la de las sesiones 7, 8 y 10.

<!-- contrato: accion=fijar-base -->

In [3]:
import os                # revisa si el archivo está en una carpeta local
import numpy as np       # cálculo con arreglos
import pandas as pd      # la librería de tablas

REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

datos = pd.read_csv(BASE + 'riesgo_impago.csv')
print('registros y columnas:', datos.shape)

registros y columnas: (6000, 16)


## Lo que el canvas fijó

El canvas resuelto de la sesión 11 dejó escrito el plan. Hoy se cumple tal como está:

| Bloque del canvas | Lo que fijó |
|:--|:--|
| Unidad y objetivo | un registro es un **cliente observado en un mes**; el objetivo, si **incumple el pago del mes siguiente** |
| Baseline | «siempre no»: no llamar a nadie |
| Métrica | sensibilidad y precisión, resumidas en un **costo en llamadas** |
| Validación | 60/20/20 estratificada, con semilla fija |
| Regla de decisión | el umbral de menor costo en validación; un impago no detectado vale **5 llamadas** |
| Candidatos | regresión logística, árbol de decisión y random forest |

<!-- contrato: accion=declarar-positiva -->

In [4]:
y = datos['incumplio_pago'].map({'no': 0, 'si': 1})  # 1 = incumple, la clase positiva

COLUMNAS_CAT = ['nivel_educacion']
COLUMNAS_NUM = []
for columna in datos.columns:  # todas menos el identificador, el objetivo y la categórica
    if columna not in ['id_cliente', 'incumplio_pago'] + COLUMNAS_CAT:
        COLUMNAS_NUM.append(columna)
X = datos[COLUMNAS_NUM + COLUMNAS_CAT]

## El protocolo, en una celda

Todo lo que el canvas fijó y que se repite en cada modelo va en **constantes al inicio**: la semilla,
el costo de un impago no detectado y la grilla de umbrales de la sesión 7. Si el canvas cambia, se
cambia acá y en ningún otro lugar.

<!-- contrato: accion=protocolo -->

In [5]:
SEMILLA = 42                                    # la misma en la partición y en cada modelo que sortea
RAZON = 5                                       # un impago no detectado vale 5 llamadas
GRILLA = np.round(np.arange(0.05, 0.91, 0.05), 2)  # umbrales de 0,05 a 0,90

### La partición 60/20/20, en dos pasos

<p align="center"></p>

Primero se aparta el 20 % de **prueba**, que no se toca hasta elegir. Después, lo que queda
(`X_trainval`) se divide en **entrenamiento** (60 % del total) y **validación** (20 %): un 25 % de
`X_trainval` es un 20 % del total.

<!-- contrato: accion=partir -->

In [6]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEMILLA, stratify=y_trainval)
display(Markdown(f'Entrenamiento: **{num(len(X_train))}** clientes · validación: **{num(len(X_val))}**, '
                 f'de los que **{num(y_val.sum())}** incumplen.'))

Entrenamiento: **3.600** clientes · validación: **1.200**, de los que **273** incumplen.

### La preparación, sin ajustar

La misma para todos los modelos: escalar las columnas numéricas y codificar la categórica. Se define
**sin ajustar**; cada modelo recibe su propia copia con `clone` y la ajusta solo con entrenamiento, dentro
de su pipeline, como en la sesión 5.

<!-- contrato: accion=preparar -->

In [7]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preparar = ColumnTransformer([
    ('num', StandardScaler(), COLUMNAS_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), COLUMNAS_CAT),
])

### La regla de decisión, en dos funciones

<p align="center"></p>

La figura es el barrido de la sesión 7 para la regresión logística, con esta misma partición: el umbral más barato fue 0,15, con un costo de 737. Esa es la fila que la logística tendrá en la tabla.

Es el barrido del umbral de la sesión 7, guardado como función para no repetirlo en cada modelo:

- `medir` cuenta, para **un** umbral, a cuántos se llama, cuántos impagos se detectan y cuántos se
  escapan, las falsas alarmas y el **costo**: 5 × impagos que se escapan + falsas alarmas.
- `umbral_de_menor_costo` recorre la grilla y devuelve la fila del umbral más barato.

<!-- contrato: accion=definir-regla -->

In [8]:
def medir(nombre, prob, y_real, umbral):
    # una fila de la tabla: qué pasa si se llama a quien alcanza el umbral
    y_real = np.asarray(y_real)
    llamar = prob >= umbral
    detectados = int((llamar & (y_real == 1)).sum())
    escapados = int(y_real.sum()) - detectados
    falsas = int((llamar & (y_real == 0)).sum())
    return {'modelo': nombre, 'umbral': float(umbral), 'llamadas': int(llamar.sum()), 'detectados': detectados,
            'escapados': escapados, 'falsas_alarmas': falsas, 'costo': RAZON * escapados + falsas}

def umbral_de_menor_costo(nombre, prob, y_real):
    # recorre la grilla y se queda con el umbral más barato
    mejor = None
    for umbral in GRILLA:
        fila = medir(nombre, prob, y_real, umbral)
        if mejor is None or fila['costo'] < mejor['costo']:
            mejor = fila
    return mejor

## Un protocolo, una tabla

### Los candidatos

El **baseline va primero**: `DummyClassifier(strategy='most_frequent')` predice siempre la clase más
frecuente, que es «no incumple», o sea, «siempre no». Después, los tres candidatos del canvas con las
configuraciones fijas, que no se reajustan hoy. **Sumar un candidato es agregar una línea.**

<!-- contrato: accion=candidatos -->

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

candidatos = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Regresión logística': LogisticRegression(max_iter=2000),
    'Árbol de decisión': DecisionTreeClassifier(max_depth=5, random_state=SEMILLA),
    'Random forest': RandomForestClassifier(min_samples_leaf=20, random_state=SEMILLA),
}

### La tabla, en validación

Cada candidato pasa por **el mismo pipeline**, se ajusta con **la misma partición** y se mide con **la
misma regla**. La tabla guarda todas las filas, también las que pierden: reportar todos los intentos es
la regla de la sesión 8. En la fila del baseline el umbral no importa: su probabilidad de incumplir es 0
para todos, así que no llama a nadie con ningún umbral.

<!-- contrato: accion=tabla -->

In [10]:
filas = []
modelos = {}
for nombre, estimador in candidatos.items():
    modelo = Pipeline([('preparar', clone(preparar)), ('modelo', clone(estimador))])
    modelo.fit(X_train, y_train)
    prob = modelo.predict_proba(X_val)[:, 1]
    filas.append(umbral_de_menor_costo(nombre, prob, y_val))
    modelos[nombre] = modelo   # se guarda para no reentrenar al elegido
tabla = pd.DataFrame(filas)
tabla

,modelo,umbral,llamadas,detectados,escapados,falsas_alarmas,costo
0,Baseline,0.05,0,0,273,0,1365
1,Regresión logística,0.15,668,216,57,452,737
2,Árbol de decisión,0.10,693,229,44,464,684
3,Random forest,0.15,665,225,48,440,680


In [11]:
costo = tabla.set_index('modelo')['costo']
display(Markdown(
    f'El baseline no llama a nadie: se le escapan los **{num(y_val.sum())}** que incumplen, y su costo es '
    f'{RAZON} × {num(y_val.sum())} = **{num(costo["Baseline"])}**. Los tres modelos cuestan cerca de la '
    f'mitad. Entre el árbol ({num(costo["Árbol de decisión"])}) y el bosque ({num(costo["Random forest"])}) '
    f'hay **{num(costo["Árbol de decisión"] - costo["Random forest"])} llamadas**. Si es una diferencia real o '
    f'la suerte de esta partición se ve en las slides, repitiendo la partición ocho veces.'
))

El baseline no llama a nadie: se le escapan los **273** que incumplen, y su costo es 5 × 273 = **1.365**. Los tres modelos cuestan cerca de la mitad. Entre el árbol (684) y el bosque (680) hay **4 llamadas**. Si es una diferencia real o la suerte de esta partición se ve en las slides, repitiendo la partición ocho veces.

## El elegido

En la clase se eligió el **random forest**, con su motivo, y recién entonces se abrió la prueba, una
sola vez (slides). Acá se toma ese pipeline, ya ajustado con entrenamiento, para guardarlo.

<!-- contrato: accion=elegir -->

In [12]:
elegido = modelos['Random forest']   # el pipeline completo: preparación y modelo

## Guardar el elegido: `joblib.dump` y `joblib.load`

Colab se reinicia, y reentrenar cada vez que se quiere rehacer un gráfico SHAP o el caso de un cliente
toma tiempo. `joblib` guarda un objeto de Python en un archivo y lo recupera:

<p align="center"></p>

- `joblib.dump(objeto, 'archivo.joblib')` **guarda** el objeto; acá, el pipeline ajustado, con la
  preparación y el modelo juntos.
- `joblib.load('archivo.joblib')` lo **recupera** tal como estaba: predice sin volver a entrenar.

Se guarda **el pipeline entero, no solo el modelo**: el modelo espera datos preparados como en
entrenamiento, y sin la preparación predice mal sin avisar.

**Dos cuidados:**

- **Solo archivos propios o de confianza:** cargar un archivo de joblib puede ejecutar código.
- **Las mismas versiones de las librerías:** con otra versión de scikit-learn, el archivo puede no
  cargar, o cargar con un aviso y dar resultados no válidos.

Colab borra los archivos de la sesión cuando se recicla el entorno. Para usarlo otro día, guárdenlo en
Drive o descárguenlo.

<!-- contrato: accion=guardar -->

In [13]:
import joblib

joblib.dump(elegido, 'riesgo_bosque.joblib')       # guarda el pipeline ajustado
cargado = joblib.load('riesgo_bosque.joblib')      # lo recupera, sin reentrenar

La comprobación: las probabilidades del pipeline cargado son **idénticas** a las del original. Se
compara en validación: este cuaderno no abre la prueba.

In [14]:
iguales = np.array_equal(elegido.predict_proba(X_val), cargado.predict_proba(X_val))
print('probabilidades idénticas tras guardar y cargar:', iguales)

probabilidades idénticas tras guardar y cargar: True


## Las versiones, anotadas

El archivo de joblib y las cifras de la tabla se repiten **con las mismas versiones de las librerías**.
Anótenlas junto a la tabla.

In [15]:
import sklearn

print('pandas', pd.__version__, '· numpy', np.__version__)
print('scikit-learn', sklearn.__version__, '· joblib', joblib.__version__)

pandas 3.0.5 · numpy 2.4.6
scikit-learn 1.9.0 · joblib 1.5.3


## Para el taller: adaptar la plantilla

Copien este cuaderno a su Drive y cambien solo lo que fija su canvas:

1. **Datos:** carguen su archivo como en el Colab de los seis casos y definan `y`, `COLUMNAS_NUM` y
   `COLUMNAS_CAT`.
2. **Protocolo:** `SEMILLA`, `RAZON` y la partición.
3. **Candidatos:** el baseline primero y sus algoritmos, una línea cada uno. En **regresión**, el
   baseline es un `DummyRegressor` y la regla se cambia por su métrica en validación.
4. **Antes de copiar una cifra:** reinicien el entorno y ejecuten todo. Solo vale la tabla que sale
   igual.

## Atribución de datos

- **Creador:** Yeh, I. C., & Lien, C. H. (2009)
- **Fuente:** [UCI Machine Learning Repository — *Default of Credit Card Clients*, dataset 350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)
- **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Modificación:** obra derivada. Se tomaron 6.000 registros y 16 columnas del original de 30.000 ×
  25, con los nombres traducidos al español en `snake_case`, la escolaridad recodificada a etiquetas y
  dos razones derivadas nuevas. Los
  atributos sexo y estado civil se excluyeron deliberadamente.